# Finer frequency bands: does splitting the EEG more finely separate AD from FTD?

The suggestion under test: use fewer features, and engineer the frequency bands, with more than the usual 5 splits.
**This is a leave-one-subject-out check, like `LOSO_ds004504.ipynb`.** Its protocol records that it was first kept private; it was published afterwards.

| model | how 1-30 Hz is divided | features |
|---|---|---|
| reference | the 95 existing band powers (5 bands x 19 channels) | 95 |
| baseline | the 5 classic bands, frontal and posterior | 10 |
| **primary** | **8 fixed bands: 1-2, 2-4, 4-6, 6-8, 8-10, 10-13, 13-20, 20-30 Hz, frontal and posterior** | **16** |
| rival 1 | 8 bands placed around each patient's own alpha frequency, plus that frequency | 17 |
| rival 2 | 8 bands whose edges are learned from the training patients only | 16 |

- **Patients are divided by nested leave-one-subject-out (LOSO).** Everything learned, including learned band edges and
  the regularisation strength, comes from the other patients.
- **5 x 5-fold is the stability check**, on the folds of Experiments 9 and 10. The result must hold under both.
- **Every patient contributes the same 300 s** of eyes-closed EEG, so recording length cannot enter the features.
- **Power is relative**, so skull thickness (and partly sex) does not enter through overall amplitude.
- **Chance is measured** by passing 1000 shuffles of the diagnoses through the whole pipeline.
- **Reads** `data/derivatives/`, `experiments_ad_vs_ftd/features/` and `experiments_ad_vs_ftd/results/c3_ds006036/`,
  `results_loso/`. **Writes** `results_loso/subbands/`.
- **Run time:** about 1 hour on 8 workers (5-7 CPU-hours), nearly all of it the permutation test in Cell 8.

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ===========================================================================
import json, os, time, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_config
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

WORK_DIR  = Path.cwd()
DATA_ROOT = WORK_DIR / "data"
EXP_DIR   = WORK_DIR / "experiments_ad_vs_ftd"
OUT_DIR   = WORK_DIR / "results_loso" / "subbands"
assert (DATA_ROOT / "participants.tsv").exists(), "no dataset: open this notebook from the project root"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- the signal: the same 300 s as the AD-vs-FTD feature table ---------------
SFREQ, EPOCH_N, N_EPOCHS = 500.0, 2000, 75       # 75 clean 4 s epochs, so spectra have 0.25 Hz bins
F_LO, F_HI = 0.5, 45.0                           # the derivatives are band-passed 0.5-45 Hz
CHANNELS = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
            "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
REGIONS  = {"frontal":   ["Fp1", "Fp2", "F3", "F4", "F7", "F8", "Fz"],
            "posterior": ["P3", "P4", "Pz", "O1", "O2"]}

# ---- the ways of dividing the spectrum, fixed before any result ---------------
CLASSIC_5 = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
FIXED_8   = [(1, 2), (2, 4), (4, 6), (6, 8), (8, 10), (10, 13), (13, 20), (20, 30)]
N_BANDS   = 8
GRID      = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 17, 20, 25, 30]    # where a learned edge may fall

M95, M5, M8, MA, ML = ("95 band powers (reference)", "5 bands x 2 regions (baseline)",
                       "8 fixed bands x 2 regions (primary)", "8 alpha-anchored bands x 2 regions + IAF",
                       "8 learned bands x 2 regions")
TESTED = [M8, MA, ML]                            # the Holm family

# ---- the pipeline of Experiments 9 and 10 ---------------------------------------
HC, AD, FTD = 0, 1, 2
SEED, N_FOLDS, N_REPEATS = 42, 5, 5
INNER_FOLDS = 5
C_GRID = [0.001, 0.01, 0.1, 1.0]
N_BOOT = 2000
N_PERM = 1000
N_JOBS = max(1, min(8, (os.cpu_count() or 2) - 2))
print(f"output : {OUT_DIR}\nworkers: {N_JOBS}")


output : C:\Projects\Major Project\results_loso\subbands
workers: 8


In [2]:
# ===========================================================================
# CELL 2 — PROTOCOL, WRITTEN BEFORE ANY SUB-BAND IS COMPARED WITH A DIAGNOSIS
# ===========================================================================
PROTOCOL = {
    "status": "private; not for the README, the deck or the report; reported whatever it shows",
    "question": "Do finer frequency bands, with far fewer features, separate AD (36) from FTD (23)?",
    "data": "ds004504 eyes closed, derivatives; every patient's first 75 clean 4 s epochs (300 s, no ASR join)",
    "features": "Hann-windowed spectrum per 4 s epoch (0.25 Hz bins, 0.5-45 Hz), divided by that epoch's total power, "
                "averaged over epochs and over the channels of each region (frontal Fp1 Fp2 F3 F4 F7 F8 Fz; posterior "
                "P3 P4 Pz O1 O2); a band's feature is log10 of its summed relative power",
    "division_of_frequencies": {
        "primary": "8 fixed bands 1-2, 2-4, 4-6, 6-8, 8-10, 10-13, 13-20, 20-30 Hz x 2 regions = 16 features",
        "rival_alpha_anchored": "edges 1, I-6, I-4, I-2, I, I+2, I+4, 20, 30 Hz, I = the patient's alpha centre of "
                                "gravity (spec_iaf_cog, 6-13 Hz, rounded to 0.25 Hz); I is also a feature (17)",
        "rival_learned": "8 contiguous bands per region with edges on the grid 1, 2, ..., 13, 15, 17, 20, 25, 30 Hz, "
                         "maximising the sum of squared two-sample t statistics between the training classes "
                         "(dynamic programming); learned inside every training set, including the inner folds (16). "
                         "The grid widens above 13 Hz because a 1 Hz grid piled edges into 22-29 Hz on shuffled "
                         "diagnoses; this was fixed before any real diagnosis was used",
        "baseline": "the 5 classic bands 0.5-4, 4-8, 8-13, 13-30, 30-45 Hz x 2 regions = 10 features",
        "reference": "the 95 existing bp_ band powers on the same 300 s",
    },
    "division_of_patients": {
        "primary": "nested leave-one-subject-out: each patient is scored by a model fitted on the task's other "
                   "patients; C from [0.001, 0.01, 0.1, 1] by 5-fold inner stratified CV AUC, inner seed 42 + fold number",
        "stability": "5 x 5-fold on the saved folds of Experiments 9 and 10, inner seed 42 + 100 x repeat + fold",
    },
    "model": "StandardScaler, then L2 logistic regression with balanced class weights (Experiment 10's pipeline)",
    "tasks": {"test": "AD vs FTD (positive class AD)",
              "controls": "AD vs HC and dementia vs HC, LOSO only, to see what fewer features cost"},
    "statistics": "patient AUC; accuracy at 0.5; 95 % CI from 2000 bootstrap resamples per class, seed 42; one-sided "
                  "permutation p from 1000 shuffles of the 59 diagnoses through the whole LOSO pipeline; Holm over "
                  "the primary and its two rivals",
    "claims": {
        "A_separates_AD_from_FTD": "Holm p < 0.05 under LOSO, and the lower 95 % bound of the same model's 5 x 5 AUC "
                                   "is above 0.5",
        "B_better_than_5_bands": "the 95 % CI of the LOSO AUC gain over the 5-band baseline is above 0",
        "C_fewer_features_cost_nothing": "AD vs HC and dementia vs HC: the lower 95 % bound of the LOSO AUC "
                                         "difference (16 features minus 95) is above -0.05",
    },
    "already_seen": "results for the 5 classic bands, the alpha centre of gravity and lagged cross-correlation (C3). "
                    "No sub-band has been compared with a diagnosis",
    "limits": "same patients as every earlier AD-vs-FTD experiment; 23 FTD patients give about 49 % power at AUC 0.65, "
              "so a null result cannot exclude a small effect; a positive result needs an independent cohort",
}

_path = OUT_DIR / "protocol.json"
if _path.exists():                    # a re-run keeps the original date and must not change the design
    _old = json.loads(_path.read_text(encoding="utf-8"))
    assert {k: v for k, v in _old.items() if k != "written"} == PROTOCOL, "protocol changed after it was written"
    PROTOCOL = _old
else:
    PROTOCOL = {"written": time.strftime("%Y-%m-%d %H:%M") + ", before any sub-band was compared with a diagnosis",
                **PROTOCOL}
    _path.write_text(json.dumps(PROTOCOL, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"protocol: {_path}  (written {PROTOCOL['written']})")


protocol: C:\Projects\Major Project\results_loso\subbands\protocol.json  (written 2026-11-18 14:36, before any sub-band was compared with a diagnosis)


In [3]:
# ===========================================================================
# CELL 3 — ONE RELATIVE SPECTRUM PER PATIENT AND REGION (no diagnosis is used)
# ===========================================================================
subjects = pd.read_csv(EXP_DIR / "features" / "subjects.csv")
table    = pd.read_csv(EXP_DIR / "features" / "eeg_features.csv")
assert list(subjects.participant_id) == [f"sub-{i:03d}" for i in range(1, 89)]
assert list(table.participant_id) == list(subjects.participant_id)

FREQS = np.fft.rfftfreq(EPOCH_N, 1 / SFREQ)
KEEP  = (FREQS >= F_LO) & (FREQS < F_HI)
FREQS = FREQS[KEEP]
HANN  = np.hanning(EPOCH_N + 1)[:-1]
R_IDX = [[CHANNELS.index(c) for c in chs] for chs in REGIONS.values()]


def relative_spectrum(ep):
    """ep: (epochs, channels, samples) -> (regions, bins). Each epoch's spectrum sums to 1, so one bad
    epoch has a bounded effect, and the result stays additive: any band is a sum of bins."""
    p = np.abs(np.fft.rfft((ep - ep.mean(-1, keepdims=True)) * HANN, axis=-1)) ** 2
    p = p[..., KEEP]
    s = (p / p.sum(-1, keepdims=True)).mean(0)
    return np.stack([s[i].mean(0) for i in R_IDX])


def subject_spectrum(sid):
    path = DATA_ROOT / "derivatives" / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*boundary.*")
        raw = mne.io.read_raw_eeglab(path, preload=False, verbose="ERROR")
    joins  = np.array([a["onset"] for a in raw.annotations if a["description"] == "boundary"]) * SFREQ
    starts = np.arange(0, raw.n_times - EPOCH_N + 1, EPOCH_N)
    inside = (joins[None, :] > starts[:, None]) & (joins[None, :] < starts[:, None] + EPOCH_N)
    starts = starts[~inside.any(axis=1)][:N_EPOCHS]
    assert len(starts) == N_EPOCHS, sid
    x = raw.get_data(picks=CHANNELS, start=0, stop=int(starts[-1]) + EPOCH_N) * 1e6
    return relative_spectrum(np.stack([x[:, s:s + EPOCH_N] for s in starts]))


def band_power(spec, lo, hi):
    """log10 relative power of [lo, hi) Hz. spec: (..., bins)."""
    return np.log10(spec[..., (FREQS >= lo) & (FREQS < hi)].sum(-1))


# ---- known answer: a 9 Hz sine must put its power in the 8-10 Hz band ----------
_t = np.arange(EPOCH_N) / SFREQ
_s = relative_spectrum(np.tile(np.sin(2 * np.pi * 9 * _t), (3, len(CHANNELS), 1)))
assert 10 ** band_power(_s, 8, 10).min() > 0.999, "spectrum failed its known-answer test"

t0 = time.time()
SPEC = np.stack(Parallel(n_jobs=N_JOBS)(delayed(subject_spectrum)(s) for s in subjects.participant_id))
assert np.allclose(SPEC.sum(-1), 1.0)

# ---- agreement with the feature table's relative power (a different estimator) ----
_chk = {b: stats.spearmanr(band_power(SPEC[:, 0], lo, hi), table[f"spec_relpow_{b}_frontal"])[0]
        for b, (lo, hi) in zip(("delta", "theta", "alpha", "beta", "gamma"), CLASSIC_5)}
assert min(_chk.values()) > 0.9, _chk
print(f"{SPEC.shape[0]} patients x {SPEC.shape[1]} regions x {SPEC.shape[2]} bins in {time.time() - t0:.0f} s")
print("rank agreement with spec_relpow (frontal):", {k: round(v, 3) for k, v in _chk.items()})


88 patients x 2 regions x 178 bins in 5 s
rank agreement with spec_relpow (frontal): {'delta': np.float64(0.932), 'theta': np.float64(0.928), 'alpha': np.float64(0.967), 'beta': np.float64(0.949), 'gamma': np.float64(0.951)}


In [4]:
# ===========================================================================
# CELL 4 — THE FEATURE SETS
# ===========================================================================
def fixed_features(bands):
    return np.column_stack([band_power(SPEC[:, r], lo, hi) for r in range(SPEC.shape[1]) for lo, hi in bands])


IAF = np.round(table.spec_iaf_cog.to_numpy() * 4) / 4        # no missing values; 7.75-10 Hz
assert not np.isnan(IAF).any() and IAF.min() - 6 > 1


def anchored_features():
    rows = []
    for spec, i in zip(SPEC, IAF):
        e = [1, i - 6, i - 4, i - 2, i, i + 2, i + 4, 20, 30]
        rows.append([band_power(spec[r], a, b) for r in range(len(spec)) for a, b in zip(e[:-1], e[1:])] + [i])
    return np.array(rows)


_bp = [c for c in table.columns if c.startswith("bp_")]
assert len(_bp) == 95
FEATURES = {M95: (table[_bp].to_numpy(float), False),
            M5:  (fixed_features(CLASSIC_5), False),
            M8:  (fixed_features(FIXED_8), False),
            MA:  (anchored_features(), False),
            ML:  (SPEC.reshape(len(SPEC), -1), True)}          # bands are learned inside each training set
for n, (x, learned) in FEATURES.items():
    assert np.isfinite(x).all(), n
    print(f"{n:44s} {N_BANDS * 2 if learned else x.shape[1]:3d} features")


95 band powers (reference)                    95 features
5 bands x 2 regions (baseline)                10 features
8 fixed bands x 2 regions (primary)           16 features
8 alpha-anchored bands x 2 regions + IAF      17 features
8 learned bands x 2 regions                   16 features


In [5]:
# ===========================================================================
# CELL 5 — THE PIPELINE (identity check against the saved LOSO run of C3)
# ===========================================================================
ATOMS = np.stack([(FREQS >= a) & (FREQS < b) for a, b in zip(GRID[:-1], GRID[1:])]).astype(float)
_IU   = np.triu_indices(len(GRID), 1)


def best_cuts(cum, y, k=N_BANDS):
    """Grid indices of the k contiguous bands with the largest summed squared t statistic (dynamic programming)."""
    n = cum.shape[1]
    L = np.log10(cum[:, _IU[1]] - cum[:, _IU[0]])
    a, b = L[y], L[~y]
    pooled = ((len(a) - 1) * a.var(0, ddof=1) + (len(b) - 1) * b.var(0, ddof=1)) / (len(a) + len(b) - 2)
    score = np.full((n, n), -np.inf)
    score[_IU] = (a.mean(0) - b.mean(0)) ** 2 / (pooled * (1 / len(a) + 1 / len(b)))
    best, back = np.full(n, -np.inf), []
    best[0] = 0.0
    for _ in range(k):
        m = best[:, None] + score
        back.append(m.argmax(0))
        best = m.max(0)
    cuts = [n - 1]
    for b_ in reversed(back):
        cuts.append(int(b_[cuts[-1]]))
    return np.array(cuts[::-1])


class LearnedBands(BaseEstimator, TransformerMixin):
    """Learns N_BANDS bands per region from the labels it is fitted on, and returns their log10 relative power."""

    def _cum(self, X):
        return [np.column_stack([np.zeros(len(r)), np.cumsum(r @ ATOMS.T, axis=1)])
                for r in np.split(X, len(REGIONS), axis=1)]

    def fit(self, X, y):
        y = np.asarray(y).astype(bool)
        self.cuts_ = [best_cuts(c, y) for c in self._cum(X)]
        return self

    def transform(self, X):
        return np.column_stack([np.log10(c[:, k[1:]] - c[:, k[:-1]]) for c, k in zip(self._cum(X), self.cuts_)])


# ---- known answer: a class difference planted in 6-8 Hz must produce cuts at 6 and 8 Hz ----
_r = np.random.default_rng(0)
_x = np.exp(_r.normal(0, 0.05, (60, 2, FREQS.size))) / FREQS.size
_y = np.arange(60) < 30
_x[np.ix_(_y, [0, 1], (FREQS >= 6) & (FREQS < 8))] *= 2
for _k in LearnedBands().fit(_x.reshape(60, -1), _y).cuts_:
    assert {GRID.index(6), GRID.index(8)} <= set(_k.tolist()), _k
    assert len(_k) == N_BANDS + 1 and _k[0] == 0 and _k[-1] == len(GRID) - 1, _k


def run_fold(X, y, train, test, seed, learned=False):
    """Experiment 10's pipeline: C tuned by inner CV on `train`, P(positive class) for `test`."""
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)
    steps = [StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)]
    pipe  = make_pipeline(*([LearnedBands()] if learned else []), *steps)
    est   = GridSearchCV(pipe, {"logisticregression__C": C_GRID}, cv=inner, scoring="roc_auc")
    return est.fit(X[train], y[train]).predict_proba(X[test])[:, 1]


def run_loso(X, y, idx, learned=False):
    q = np.full(len(y), np.nan)
    for k, i in enumerate(idx, start=1):
        q[i] = run_fold(X, y, idx[idx != i], np.array([i]), SEED + k, learned)[0]
    return q


LABELS = subjects.label.to_numpy()
_mask  = {"AD vs FTD": (LABELS == AD, LABELS == FTD), "AD vs HC": (LABELS == AD, LABELS == HC),
          "Dementia vs HC": (LABELS != HC, LABELS == HC)}
TASKS  = {t: dict(y=p.astype(int), idx=np.flatnonzero(p | n), pos=np.flatnonzero(p), neg=np.flatnonzero(n))
          for t, (p, n) in _mask.items()}
T_FTD  = TASKS["AD vs FTD"]
assert (len(T_FTD["pos"]), len(T_FTD["neg"])) == (36, 23)

with np.load(EXP_DIR / "features" / "c3_ds006036_matrices.npz") as z:
    _c3 = z["xcz_06"]
_saved = np.load(WORK_DIR / "results_loso" / "c3_oof.npz", allow_pickle=True)
_q0 = _saved["q_ad"][list(_saved["runs"]).index("ds006036, 44 s (primary)")]
_q1 = run_loso(_c3, T_FTD["y"], T_FTD["idx"])
assert np.allclose(_q1[T_FTD["idx"]], _q0[T_FTD["idx"]], atol=1e-10), "the pipeline differs from the saved LOSO run"
print("identity check passed: this pipeline reproduces the saved LOSO scores of C3 exactly")


identity check passed: this pipeline reproduces the saved LOSO scores of C3 exactly


In [6]:
# ===========================================================================
# CELL 6 — EVERY MODEL UNDER LOSO, AND AD VS FTD UNDER 5 x 5
# ===========================================================================
def loso_job(t, m):
    X, learned = FEATURES[m]
    return t, m, run_loso(X, TASKS[t]["y"], TASKS[t]["idx"], learned)


_fold_of = np.load(EXP_DIR / "results" / "c3_ds006036" / "c3_oof.npz", allow_pickle=True)["fold_of"]


def five_by_five(m):
    X, learned = FEATURES[m]
    y, idx = T_FTD["y"], T_FTD["idx"]
    q = np.full((N_REPEATS, len(y)), np.nan)
    for r in range(N_REPEATS):
        for f in range(1, N_FOLDS + 1):
            test = idx[_fold_of[r, idx] == f]
            q[r, test] = run_fold(X, y, np.setdiff1d(idx, test), test, SEED + 100 * (r + 1) + f, learned)
    return m, q


t0 = time.time()
with parallel_config(backend="loky", inner_max_num_threads=1):
    _a = Parallel(n_jobs=N_JOBS)(delayed(loso_job)(t, m) for t in TASKS for m in FEATURES)
    _b = Parallel(n_jobs=N_JOBS)(delayed(five_by_five)(m) for m in FEATURES)
LOSO = {t: {m: q for tt, m, q in _a if tt == t} for t in TASKS}
FIVE = dict(_b)
print(f"{len(_a)} LOSO runs and {len(_b)} 5 x 5 runs in {time.time() - t0:.0f} s")


15 LOSO runs and 5 5 x 5 runs in 25 s


In [7]:
# ===========================================================================
# CELL 7 — RESULTS: AUC, ACCURACY AND THE GAIN OVER EACH REFERENCE
# ===========================================================================
def auc(pos, neg):
    d = pos[..., :, None] - neg[..., None, :]
    return (d > 0).mean((-2, -1)) + 0.5 * (d == 0).mean((-2, -1))


_rng = np.random.default_rng(SEED)
BOOT_IDX = {t: (_rng.choice(T["pos"], (N_BOOT, len(T["pos"]))), _rng.choice(T["neg"], (N_BOOT, len(T["neg"]))))
            for t, T in TASKS.items()}
lo_hi = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))


def evaluate(q, t):
    q = np.atleast_2d(q)
    T, (bp, bn) = TASKS[t], BOOT_IDX[t]
    np_, nn = len(T["pos"]), len(T["neg"])
    sens, spec = (q[:, T["pos"]] >= 0.5).mean(-1), (q[:, T["neg"]] < 0.5).mean(-1)
    b_auc  = auc(q[:, bp], q[:, bn]).mean(0)
    b_sens, b_spec = (q[:, bp] >= 0.5).mean(-1), (q[:, bn] < 0.5).mean(-1)
    b_acc  = ((b_sens * np_ + b_spec * nn) / (np_ + nn)).mean(0)
    row = dict(auc=auc(q[:, T["pos"]], q[:, T["neg"]]).mean(), acc=((sens * np_ + spec * nn) / (np_ + nn)).mean(),
               recall_pos=sens.mean(), recall_neg=spec.mean())
    row["bal_acc"] = (row["recall_pos"] + row["recall_neg"]) / 2
    row["auc_lo"], row["auc_hi"] = lo_hi(b_auc)
    row["acc_lo"], row["acc_hi"] = lo_hi(b_acc)
    return row, b_auc


def boot_p(d):
    return min(1.0, 2 * (min((d <= 0).sum(), (d >= 0).sum()) + 1) / (len(d) + 1))


rows, gains, BOOTS = [], [], {}
for t in TASKS:
    for m, q in LOSO[t].items():
        r, BOOTS[t, m] = evaluate(q, t)
        n = len(TASKS[t]["idx"])
        rows.append(dict(task=t, split="LOSO", model=m, correct=f"{round(r['acc'] * n)}/{n}", **r))
    for m in (M8, MA, ML):
        for ref in (M5, M95):
            d = BOOTS[t, m] - BOOTS[t, ref]
            real = auc(LOSO[t][m][TASKS[t]["pos"]], LOSO[t][m][TASKS[t]["neg"]]) \
                 - auc(LOSO[t][ref][TASKS[t]["pos"]], LOSO[t][ref][TASKS[t]["neg"]])
            gains.append(dict(task=t, model=m, minus=ref, d_auc=float(real), lo=lo_hi(d)[0], hi=lo_hi(d)[1], p=boot_p(d)))
for m, q in FIVE.items():
    r, _ = evaluate(q, "AD vs FTD")
    rows.append(dict(task="AD vs FTD", split="5 x 5", model=m, correct="", **r))
RESULTS, GAINS = pd.DataFrame(rows), pd.DataFrame(gains)

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 60)
for (t, s), g in RESULTS.groupby(["task", "split"], sort=False):
    print(f"\n{t}, {s}")
    for _, r in g.iterrows():
        print(f"  {r.model:44s} AUC {r.auc:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}]   accuracy {r.acc:.3f} "
              f"[{r.acc_lo:.3f}, {r.acc_hi:.3f}] {r.correct:>6s}   balanced {r.bal_acc:.3f}")
print("\nLOSO AUC gain (paired bootstrap)\n")
print(GAINS.round(3).to_string(index=False))



AD vs FTD, LOSO
  95 band powers (reference)                   AUC 0.588 [0.431, 0.742]   accuracy 0.542 [0.424, 0.661]  32/59   balanced 0.539
  5 bands x 2 regions (baseline)               AUC 0.399 [0.255, 0.551]   accuracy 0.390 [0.271, 0.508]  23/59   balanced 0.367
  8 fixed bands x 2 regions (primary)          AUC 0.389 [0.233, 0.542]   accuracy 0.441 [0.322, 0.559]  26/59   balanced 0.416
  8 alpha-anchored bands x 2 regions + IAF     AUC 0.419 [0.261, 0.575]   accuracy 0.458 [0.322, 0.593]  27/59   balanced 0.446
  8 learned bands x 2 regions                  AUC 0.457 [0.304, 0.618]   accuracy 0.458 [0.339, 0.593]  27/59   balanced 0.454

AD vs HC, LOSO
  95 band powers (reference)                   AUC 0.796 [0.680, 0.898]   accuracy 0.738 [0.631, 0.846]  48/65   balanced 0.734
  5 bands x 2 regions (baseline)               AUC 0.750 [0.623, 0.863]   accuracy 0.708 [0.600, 0.815]  46/65   balanced 0.703
  8 fixed bands x 2 regions (primary)          AUC 0.827 [0.718, 0.920]

In [8]:
# ===========================================================================
# CELL 8 — CHANCE: THE WHOLE LOSO PIPELINE ON SHUFFLED DIAGNOSES (primary and both rivals)
# ===========================================================================
def shuffled_aucs(k):
    idx = T_FTD["idx"]
    y = T_FTD["y"].copy()
    y[idx] = np.random.default_rng(SEED + 30_000 + k).permutation(y[idx])
    out = []
    for m in TESTED:
        X, learned = FEATURES[m]
        q = run_loso(X, y, idx, learned)
        out.append(auc(q[idx[y[idx] == 1]], q[idx[y[idx] == 0]]))
    return out


t0 = time.time()
with parallel_config(backend="loky", inner_max_num_threads=1):
    NULL = np.array(Parallel(n_jobs=N_JOBS)(delayed(shuffled_aucs)(k) for k in range(N_PERM)))
print(f"{len(NULL)} shuffles x {len(TESTED)} models in {(time.time() - t0) / 60:.1f} min")

_obs = RESULTS[(RESULTS.task == "AD vs FTD") & (RESULTS.split == "LOSO")].set_index("model")
_five = RESULTS[(RESULTS.task == "AD vs FTD") & (RESULTS.split == "5 x 5")].set_index("model")
perm = []
for j, m in enumerate(TESTED):
    a = _obs.loc[m, "auc"]
    perm.append(dict(model=m, auc_loso=a, null_median=np.median(NULL[:, j]), null_lo=np.percentile(NULL[:, j], 2.5),
                     null_hi=np.percentile(NULL[:, j], 97.5), p=(1 + (NULL[:, j] >= a).sum()) / (1 + len(NULL)),
                     auc_5x5=_five.loc[m, "auc"], auc_5x5_lo=_five.loc[m, "auc_lo"]))
PERM = pd.DataFrame(perm).sort_values("p")
PERM["p_holm"] = np.maximum.accumulate((PERM.p * np.arange(len(PERM), 0, -1)).clip(upper=1.0))
PERM = PERM.set_index("model").loc[TESTED]
print(PERM.round(3).to_string())


1000 shuffles x 3 models in 51.0 min
                                          auc_loso  null_median  null_lo  null_hi      p  auc_5x5  auc_5x5_lo  p_holm
model                                                                                                                
8 fixed bands x 2 regions (primary)          0.389        0.498    0.285    0.676  0.858    0.412       0.296     1.0
8 alpha-anchored bands x 2 regions + IAF     0.419        0.499    0.272    0.679  0.777    0.484       0.363     1.0
8 learned bands x 2 regions                  0.457        0.493    0.277    0.679  0.658    0.477       0.352     1.0


In [9]:
# ===========================================================================
# CELL 9 — THE THREE PRE-REGISTERED CLAIMS, THE BARS AND THE CONFOUNDERS
# ===========================================================================
_g = GAINS.set_index(["task", "model", "minus"])
claims = []
for m in TESTED:
    a = bool(PERM.loc[m, "p_holm"] < 0.05 and PERM.loc[m, "auc_5x5_lo"] > 0.5)
    b = bool(_g.loc[("AD vs FTD", m, M5), "lo"] > 0)
    c = all(_g.loc[(t, m, M95), "lo"] > -0.05 for t in ("AD vs HC", "Dementia vs HC"))
    claims.append(dict(model=m, A_separates_AD_from_FTD=a, B_better_than_5_bands=b, C_fewer_features_cost_nothing=c))
CLAIMS = pd.DataFrame(claims).set_index("model")
print(CLAIMS.to_string())


def verdict(bar, lo, hi):
    return "level" if lo <= bar <= hi else "ahead" if bar < lo else "behind"


BARS = [("AD vs FTD", "auc", "Mlinaric et al. 2026, nested LOSO", 0.651), ("AD vs FTD", "auc", "Wang et al. 2026, LOSO", 0.823),
        ("AD vs FTD", "auc", "our C3, LOSO (private)", 0.574), ("AD vs HC", "auc", "Wang et al. 2026, LOSO", 0.912),
        ("AD vs HC", "acc", "Wang et al. 2026, LOSO", 0.877), ("AD vs HC", "auc", "our GAT, LOSO (private)", 0.915),
        ("Dementia vs HC", "auc", "our soft vote, 3 x 5-fold", 0.898),
        ("Dementia vs HC", "acc", "our soft vote, 3 x 5-fold", 0.826),
        ("Dementia vs HC", "acc", "Stefanou et al. 2025, per epoch", 0.807)]
_p = RESULTS[(RESULTS.split == "LOSO") & (RESULTS.model == M8)].set_index("task")
VERDICTS = pd.DataFrame([dict(task=t, metric=k, bar=n, published=v, ours=_p.loc[t, k], lo=_p.loc[t, f"{k}_lo"],
                              hi=_p.loc[t, f"{k}_hi"], verdict=verdict(v, _p.loc[t, f"{k}_lo"], _p.loc[t, f"{k}_hi"]))
                         for t, k, n, v in BARS])
print("\nthe primary model against the bars (level = the bar lies inside our 95 % CI)\n")
print(VERDICTS.round(3).to_string(index=False))

# ---- non-EEG information alone, under the same LOSO ---------------------------
_sex, _age = (subjects.sex == "F").to_numpy(float), subjects.age.to_numpy(float)
NON_EEG = {"sex": _sex[:, None], "age": _age[:, None], "age + sex": np.column_stack([_age, _sex]),
           "recording length": subjects.duration_s.to_numpy(float)[:, None]}
_n = []
for n, x in NON_EEG.items():
    r, _ = evaluate(run_loso(x, T_FTD["y"], T_FTD["idx"]), "AD vs FTD")
    _n.append(dict(input=n, auc=r["auc"], lo=r["auc_lo"], hi=r["auc_hi"]))
NON_EEG_T = pd.DataFrame(_n)
print("\nAD vs FTD from non-EEG information alone (never a model input above)\n")
print(NON_EEG_T.round(3).to_string(index=False))


# ---- does the primary score add anything to MMSE + age + sex? --------------------
def loglik(Xc, y):
    m = LogisticRegression(C=np.inf, max_iter=10000).fit(Xc, y)
    p = np.clip(m.predict_proba(Xc)[:, 1], 1e-12, 1 - 1e-12)
    return (y * np.log(p) + (1 - y) * np.log(1 - p)).sum()


_s = subjects.iloc[T_FTD["idx"]]
_yd = T_FTD["y"][T_FTD["idx"]]
_base = StandardScaler().fit_transform(np.column_stack([_s.mmse, _s.age, _s.sex == "F"]).astype(float))
_q = np.clip(LOSO["AD vs FTD"][M8][T_FTD["idx"]], 1e-6, 1 - 1e-6)
_l = np.log(_q / (1 - _q))
LR_CHI2 = 2 * (loglik(np.column_stack([_base, (_l - _l.mean()) / _l.std()]), _yd) - loglik(_base, _yd))
P_BEYOND = float(stats.chi2.sf(LR_CHI2, 1))
print(f"\nprimary score beyond MMSE + age + sex: likelihood-ratio chi2 {LR_CHI2:.2f}, p = {P_BEYOND:.3f}")

# ---- where the learned edges fall, over the LOSO training sets (descriptive) -------
_cum = LearnedBands()._cum(FEATURES[ML][0])
_idx = T_FTD["idx"]
_counts = np.zeros((len(REGIONS), len(GRID)))
for i in _idx:
    tr = _idx[_idx != i]
    for r, c in enumerate(_cum):
        _counts[r, best_cuts(c[tr], T_FTD["y"][tr].astype(bool))] += 1
EDGES = pd.DataFrame(_counts[:, 1:-1].T / len(_idx), columns=list(REGIONS), index=[f"{f} Hz" for f in GRID[1:-1]])
print("\nshare of LOSO training sets that put a band edge at each frequency (edges in at least half of them)\n")
print(EDGES[(EDGES >= 0.5).any(axis=1)].round(2).to_string())


                                          A_separates_AD_from_FTD  B_better_than_5_bands  C_fewer_features_cost_nothing
model                                                                                                                  
8 fixed bands x 2 regions (primary)                         False                  False                          False
8 alpha-anchored bands x 2 regions + IAF                    False                  False                          False
8 learned bands x 2 regions                                 False                  False                          False

the primary model against the bars (level = the bar lies inside our 95 % CI)

          task metric                               bar  published  ours    lo    hi verdict
     AD vs FTD    auc Mlinaric et al. 2026, nested LOSO      0.651 0.389 0.233 0.542  behind
     AD vs FTD    auc            Wang et al. 2026, LOSO      0.823 0.389 0.233 0.542  behind
     AD vs FTD    auc            our C3, L


AD vs FTD from non-EEG information alone (never a model input above)

           input   auc    lo    hi
             sex 0.406 0.246 0.580
             age 0.537 0.382 0.684
       age + sex 0.616 0.464 0.757
recording length 0.652 0.493 0.797

primary score beyond MMSE + age + sex: likelihood-ratio chi2 0.02, p = 0.876

share of LOSO training sets that put a band edge at each frequency (edges in at least half of them)

       frontal  posterior
2 Hz      0.27       1.00
3 Hz      1.00       1.00
4 Hz      1.00       0.78
5 Hz      1.00       0.58
6 Hz      1.00       0.78
7 Hz      0.88       0.02
8 Hz      0.53       0.02
9 Hz      0.58       0.44
17 Hz     0.36       0.83
20 Hz     0.36       0.78


In [10]:
# ===========================================================================
# CELL 10 — SAVE
# ===========================================================================
np.savez_compressed(OUT_DIR / "oof.npz", subjects=subjects.participant_id.to_numpy(), labels=LABELS,
                    models=np.array(list(FEATURES)), tasks=np.array(list(TASKS)),
                    loso=np.stack([np.stack([LOSO[t][m] for m in FEATURES]) for t in TASKS]),
                    five_by_five=np.stack([FIVE[m] for m in FEATURES]), null_auc=NULL, tested=np.array(TESTED),
                    spectra=SPEC, freqs=FREQS)
RESULTS.to_csv(OUT_DIR / "results.csv", index=False)
GAINS.to_csv(OUT_DIR / "gains.csv", index=False)
PERM.to_csv(OUT_DIR / "permutation.csv")
CLAIMS.to_csv(OUT_DIR / "claims.csv")
VERDICTS.to_csv(OUT_DIR / "verdicts.csv", index=False)
NON_EEG_T.to_csv(OUT_DIR / "non_eeg.csv", index=False)
EDGES.to_csv(OUT_DIR / "learned_edges.csv")
SUMMARY = dict(protocol=PROTOCOL["written"], n_perm=N_PERM,
               ad_vs_ftd={m: {k: round(float(v), 4) for k, v in PERM.loc[m].items()} for m in TESTED},
               claims=CLAIMS.to_dict(orient="index"),
               primary_beyond_mmse_age_sex=dict(chi2=round(float(LR_CHI2), 3), p=round(P_BEYOND, 4)))
(OUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=float), encoding="utf-8")
print(f"saved to {OUT_DIR}:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


saved to C:\Projects\Major Project\results_loso\subbands: claims.csv, gains.csv, learned_edges.csv, non_eeg.csv, oof.npz, permutation.csv, protocol.json, results.csv, summary.json, verdicts.csv
